# SparkOCR-VLM — Local Quickstart

Run this notebook on your Intel Mac. Mock backend by default; swap to OpenRouter at the end.

In [ ]:
# 1. Install (only the first time)
# %pip install -e ..

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve() / 'src'))

## 2. Build local Spark

In [ ]:
from sparkocr_vlm.utils.spark_helpers import build_local_spark
spark = build_local_spark(cores='2', memory='2g')
spark

## 3. Generate synthetic PDFs

In [ ]:
from pathlib import Path
from tests.harness.synthetic_pdf import SyntheticPDFBuilder
from tests.harness.golden import install_mock_table_from_fixtures

FIX = Path('../tests/fixtures')
SyntheticPDFBuilder(out_dir=FIX).build_all()
install_mock_table_from_fixtures(FIX)
list(FIX.glob('*.pdf'))

## 4. Run pipeline with mock backend

In [ ]:
from sparkocr_vlm import OCRPipeline

pipeline = OCRPipeline(backend='mock', model='mock-model',
                       input_path=str(FIX), output_path='./_out_quickstart')
df = pipeline.run(spark)
df.toPandas()

## 5. Single-doc convenience path (no Spark)

In [ ]:
from sparkocr_vlm import OCRPipeline
single = OCRPipeline(backend='mock').parse_single(FIX / 'synth_invoice.pdf')
single

## 6. Swap to OpenRouter (requires OPENROUTER_API_KEY)

In [ ]:
# import os
# os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1-...'
# real = OCRPipeline(backend='openrouter', model='deepseek-ai/DeepSeek-OCR-v2:free').parse_single(FIX / 'synth_invoice.pdf')
# print(real.iloc[0]['markdown'])